In [1]:
import pandas as pd
import pysubgroup as ps
from itertools import product

Análise comparativa entre as tabelas dos arquivos datatran2007.csv e datatran2025.csv (agrupamento por ocorrência)

### 1. Carregando os dados

In [ ]:
colunas_em_comum = ['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km','municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 
    'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 
    'feridos', 'veiculos'
]

df07 = pd.read_csv("datatran2007.csv",sep=";",encoding="latin1", usecols=colunas_em_comum)                  
df25 = pd.read_csv("datatran2025.csv",sep=";",encoding="latin1", usecols=colunas_em_comum)


### 2. Pré-processamento

In [3]:
def preprocess(df):
    df = df.copy()

    # Target binário: acidente com vítima fatal
    df['fatal'] = (df['mortos'] > 0).astype(int)

    # Turno do dia a partir do horário
    df['horario'] = df['horario'].astype(str).str[:2]
    df['horario'] = pd.to_numeric(df['horario'], errors='coerce')
    df['turno'] = pd.cut(
        df['horario'],
        bins=[-1, 5, 11, 17, 23],
        labels=['madrugada', 'manha', 'tarde', 'noite']
    )

    # Colunas categóricas que usaremos como features
    cats = [
        'dia_semana', 'uf', 'causa_acidente', 'tipo_acidente',
        'fase_dia', 'condicao_metereologica', 'tipo_pista',
        'tracado_via', 'uso_solo', 'turno'
    ]
    for c in cats:
        df[c] = df[c].astype(str).str.strip().str.lower()

    df = df.dropna(subset=['fatal'] + cats)
    return df, cats

df07_clean, cats = preprocess(df07)
df25_clean, _    = preprocess(df25)

### 3. Descoberta de subgrupos

In [ ]:
def run_sd(df, cats, target_col='fatal', top_k=20, depth=3):
    """Retorna top-K subgrupos ordenados por WRAcc."""
    target = ps.BinaryTarget(target_col, True)

    searchspace = ps.create_selectors(df, ignore=[target_col])
    # Filtra para usar só as colunas categóricas que preparamos
    searchspace = [s for s in searchspace
                   if any(c in str(s) for c in cats)]

    task = ps.SubgroupDiscoveryTask(
        df,
        target,
        searchspace,
        result_set_size=top_k,
        depth=depth,
        qf=ps.WRAccQF()          # Weighted Relative Accuracy
    )

    result = ps.BeamSearch().execute(task)
    return result.to_dataframe()

print("Rodando SD em 2007...")
res07 = run_sd(df07_clean, cats)

print("Rodando SD em 2025...")
res25 = run_sd(df25_clean, cats)

### 4. Comparação temporal

In [11]:
def subgroup_to_str(sg):
    return str(sg).strip().lower()

# Mapeia string -> linha original para recuperar métricas
map07 = {subgroup_to_str(r['subgroup']): r for _, r in res07.iterrows()}
map25 = {subgroup_to_str(r['subgroup']): r for _, r in res25.iterrows()}

set07 = set(map07.keys())
set25 = set(map25.keys())

jaccard = len(set07 & set25) / len(set07 | set25) if set07 | set25 else 0

def make_table(keys, map_principal, map_secundario=None):
    rows = []
    for k in sorted(keys):
        r = map_principal[k]
        row = {
            'subgrupo': k,
            'quality':  round(r['quality'], 4),
            'tamanho':  int(r['size_sg']),
            'p(fatal)': round(r['target_share_sg'], 4),
        }
        if map_secundario and k in map_secundario:
            r2 = map_secundario[k]
            row['quality_outro_ano'] = round(r2['quality'], 4)
            row['p(fatal)_outro_ano'] = round(r2['target_share_sg'], 4)
        rows.append(row)
    return pd.DataFrame(rows)

df_persistentes = make_table(set07 & set25, map07, map25)
df_novos        = make_table(set25 - set07, map25)
df_extintos     = make_table(set07 - set25, map07)

# Renomeia colunas dos persistentes para deixar claro qual ano é qual
df_persistentes = df_persistentes.rename(columns={
    'quality':           'quality_2007',
    'tamanho':           'tamanho_2007',
    'p(fatal)':          'p(fatal)_2007',
    'quality_outro_ano': 'quality_2025',
    'p(fatal)_outro_ano':'p(fatal)_2025',
})

sep = "-" * 60

print(f"\n{'=' * 60}")
print(f"  COMPARAÇÃO TEMPORAL  |  Jaccard: {jaccard:.2f}")
print(f"{'=' * 60}")

print(f"\n── PERSISTENTES ({len(df_persistentes)}) ─────────────────────────────")
print(df_persistentes.to_string(index=False))

print(f"\n── NOVOS em 2025 ({len(df_novos)}) ──────────────────────────────────")
print(df_novos.to_string(index=False))

print(f"\n── EXTINTOS desde 2007 ({len(df_extintos)}) ─────────────────────────")
print(df_extintos.to_string(index=False))

print(f"\n{sep}")
print(f"  Persistentes: {len(df_persistentes):>3}  |  "
      f"Novos: {len(df_novos):>3}  |  "
      f"Extintos: {len(df_extintos):>3}")
print(sep)


  COMPARAÇÃO TEMPORAL  |  Jaccard: 0.18

── PERSISTENTES (6) ─────────────────────────────
                                                  subgrupo  quality_2007  tamanho_2007  p(fatal)_2007  quality_2025  p(fatal)_2025
                                   fase_dia=='plena noite'        0.0064         42871         0.0618        0.0102         0.1018
         fase_dia=='plena noite' and tipo_pista=='simples'        0.0063         24751         0.0751        0.0101         0.1315
                          tipo_acidente=='colisão frontal'        0.0077          4581         0.2565        0.0146         0.2946
tipo_acidente=='colisão frontal' and tipo_pista=='simples'        0.0073          3999         0.2761        0.0141         0.3128
                                     tipo_pista=='simples'        0.0069         71637         0.0548        0.0128         0.0986
             tipo_pista=='simples' and tracado_via=='reta'        0.0062         48402         0.0589        0.0079       

### Exportação para análise

In [6]:
res07['ano'] = 2007
res25['ano'] = 2025
comparativo = pd.concat([res07, res25], ignore_index=True)
comparativo.to_csv("subgrupos_comparativo.csv", index=False)

print("\nTop 5 subgrupos 2007:")
print(res07[['subgroup', 'quality', 'size_sg', 'target_share_sg']].head())
print("\nTop 5 subgrupos 2025:")
print(res25[['subgroup', 'quality', 'size_sg', 'target_share_sg']].head())


Top 5 subgrupos 2007:
                                            subgroup   quality  size_sg  \
0        tipo_pista=='simples' AND uso_solo=='rural'  0.007717    46440   
1                   tipo_acidente=='colisão frontal'  0.007674     4581   
2  tipo_acidente=='colisão frontal' AND tipo_pist...  0.007312     3999   
3                              tipo_pista=='simples'  0.006851    71637   
4           tipo_acidente=='atropelamento de pessoa'  0.006821     3852   

   target_share_sg  
0         0.063824  
1         0.256494  
2         0.276069  
3         0.054818  
4         0.268692  

Top 5 subgrupos 2025:
                                            subgroup   quality  size_sg  \
0          tipo_pista=='simples' AND uso_solo=='não'  0.015581    23860   
1                   tipo_acidente=='colisão frontal'  0.014554     4739   
2  tipo_acidente=='colisão frontal' AND tipo_pist...  0.014073     4236   
3  tipo_acidente=='colisão frontal' AND uso_solo=...  0.012943     3359   
4 